[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week05_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 5 Assignment: CUPED Implementation

## QuickCart — Cutting Experiment Duration with Pre-Experiment Data

QuickCart wants to detect a **2% lift in weekly revenue per user**. With the current approach (standard t-test), achieving 80% power requires **4 weeks** of data collection. The product team is frustrated — that's a full month before they know if a change works.

Your task: implement **CUPED (Controlled-experiment Using Pre-Experiment Data)**, a variance reduction technique that uses each user's pre-experiment behavior as a covariate. If pre-experiment revenue is correlated with in-experiment revenue, CUPED can dramatically reduce variance and thus the required sample size.

**Key idea:** If we know a user spent $100/week before the experiment, their in-experiment spending of $102 is much more informative than the raw $102 alone. CUPED formalizes this intuition.

**The CUPED adjustment:**

$$Y_{\text{cuped}} = Y - \theta \cdot X$$

where:
- $Y$ = metric during the experiment (pilot period)
- $X$ = metric during the pre-experiment period
- $\theta = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}$

The variance reduction factor is:

$$\text{Var}(Y_{\text{cuped}}) = \text{Var}(Y) \cdot (1 - \rho^2)$$

where $\rho$ is the correlation between $X$ and $Y$.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(42)

---

## Task 1: Compute Per-User Metric for a Given Period

### Context

QuickCart's raw data has one row per transaction: `(user_id, date, revenue)`. Before applying CUPED, you need a function that aggregates this into **one metric value per user** for a specified date range.

Users who appear in `list_user_id` but have no transactions in the period should get a value of **0** (they were active but didn't purchase).

### Problem

Implement `calculate_metric` that:
1. Filters the DataFrame to rows within the specified period (`begin <= date <= end`)
2. Groups by user and sums the value column
3. Ensures every user in `list_user_id` appears in the output (fill missing users with 0)
4. Returns a DataFrame with columns `[user_id_name, metric_name]`

<details>
<summary>Hint 1 (click to expand)</summary>

Use boolean indexing to filter dates:
```python
mask = (df[date_name] >= period['begin']) & (df[date_name] <= period['end'])
```

</details>

<details>
<summary>Hint 2 (click to expand)</summary>

After groupby-sum, reindex to the full user list:
```python
result = result.reindex(list_user_id, fill_value=0).reset_index()
```
Or use `pd.merge` with a DataFrame of all user IDs and `fillna(0)`.

</details>

In [ ]:
def calculate_metric(
    df: pd.DataFrame,
    value_name: str,
    user_id_name: str,
    list_user_id: list,
    date_name: str,
    period: dict,
    metric_name: str
) -> pd.DataFrame:
    """Compute per-user metric (sum of values) for a given date period.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw transaction data with columns for user_id, date, and value.
    value_name : str
        Column name containing the numeric value to aggregate (e.g., 'revenue').
    user_id_name : str
        Column name containing the user identifier.
    list_user_id : list
        List of all user IDs that should appear in the output.
    date_name : str
        Column name containing the date.
    period : dict
        Dictionary with keys 'begin' and 'end' specifying the date range.
    metric_name : str
        Name for the output metric column.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns [user_id_name, metric_name], one row per user
        in list_user_id. Users with no transactions get 0.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test Task 1
test_df = pd.DataFrame({
    'user_id': ['A', 'A', 'B', 'B', 'C', 'A'],
    'date': pd.to_datetime(['2024-01-01', '2024-01-02', '2024-01-01', 
                            '2024-01-05', '2024-01-03', '2024-01-10']),
    'revenue': [10, 20, 15, 25, 30, 50]
})

all_users = ['A', 'B', 'C', 'D']  # D has no transactions
period = {'begin': '2024-01-01', 'end': '2024-01-03'}

result = calculate_metric(test_df, 'revenue', 'user_id', all_users, 'date', period, 'total_revenue')
print(result)

# Verify
assert len(result) == 4, "Should have one row per user in list_user_id"
assert set(result.columns) == {'user_id', 'total_revenue'}, f"Expected columns [user_id, total_revenue], got {list(result.columns)}"

result_dict = result.set_index('user_id')['total_revenue'].to_dict()
assert result_dict['A'] == 30, f"User A should have 10+20=30, got {result_dict['A']}"
assert result_dict['B'] == 15, f"User B should have 15 (only Jan 1 is in range), got {result_dict['B']}"
assert result_dict['C'] == 30, f"User C should have 30, got {result_dict['C']}"
assert result_dict['D'] == 0, f"User D should have 0 (no transactions), got {result_dict['D']}"
print("\nAll Task 1 checks passed!")

---

## Task 2: Implement CUPED-Adjusted Metric

### Context

Now that you can compute per-user metrics for any period, you'll implement the full CUPED adjustment.

The idea: use **pre-pilot period** revenue as the covariate $X$ and **pilot period** revenue as the outcome $Y$. The adjusted metric removes the predictable variation:

$$Y_{\text{cuped}} = Y - \theta \cdot (X - \bar{X})$$

where $\theta = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}$

Note: Subtracting $\bar{X}$ ensures the CUPED metric has the same mean as the original (important for interpretability), but since we compare groups, it doesn't affect the test. The simpler form $Y - \theta \cdot X$ is also acceptable for hypothesis testing.

### Problem

Implement `calculate_metric_cuped` that:
1. Computes the metric for both the pre-pilot and pilot periods (using your `calculate_metric`)
2. Calculates $\theta = \text{Cov}(Y, X) / \text{Var}(X)$
3. Computes the CUPED-adjusted metric: $Y_{\text{cuped}} = Y - \theta \cdot X$
4. Returns a DataFrame with columns: `[user_id, metric, metric_prepilot, metric_cuped]`

<details>
<summary>Hint 1 (click to expand)</summary>

Use `np.cov` to compute covariance. Note that `np.cov(a, b)` returns a 2x2 matrix:
```python
cov_matrix = np.cov(pilot_values, prepilot_values)
theta = cov_matrix[0, 1] / cov_matrix[1, 1]
```

</details>

<details>
<summary>Hint 2 (click to expand)</summary>

Merge the prepilot and pilot DataFrames on user_id, then compute:
```python
df_merged['metric_cuped'] = df_merged['metric'] - theta * df_merged['metric_prepilot']
```

</details>

In [ ]:
def calculate_metric_cuped(
    df: pd.DataFrame,
    value_name: str,
    user_id_name: str,
    list_user_id: list,
    date_name: str,
    periods: dict,
    metric_name: str
) -> pd.DataFrame:
    """Compute CUPED-adjusted metric using pre-experiment data.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw transaction data.
    value_name : str
        Column name for the numeric value.
    user_id_name : str
        Column name for user identifier.
    list_user_id : list
        List of all user IDs.
    date_name : str
        Column name for date.
    periods : dict
        Dictionary with structure:
        {
            'prepilot': {'begin': ..., 'end': ...},
            'pilot': {'begin': ..., 'end': ...}
        }
    metric_name : str
        Base name for the metric.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - user_id_name: user identifier
        - 'metric': pilot period metric value
        - 'metric_prepilot': pre-pilot period metric value
        - 'metric_cuped': CUPED-adjusted metric
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test Task 2 with synthetic data
np.random.seed(42)
n_users = 1000
user_ids = [f'user_{i}' for i in range(n_users)]

# Generate correlated pre-period and pilot-period behavior
# Users who spend more before tend to spend more during
user_base_spend = np.random.lognormal(3, 1, n_users)  # inherent spending level

# Create transaction data
rows = []
for i, uid in enumerate(user_ids):
    # Pre-pilot: 2 weeks of data
    for day in pd.date_range('2024-01-01', '2024-01-14'):
        if np.random.random() < 0.3:  # 30% chance of purchase each day
            rows.append({'user_id': uid, 'date': day, 
                        'revenue': user_base_spend[i] * np.random.exponential(1)})
    # Pilot: 2 weeks of data
    for day in pd.date_range('2024-01-15', '2024-01-28'):
        if np.random.random() < 0.3:
            rows.append({'user_id': uid, 'date': day,
                        'revenue': user_base_spend[i] * np.random.exponential(1)})

df_transactions = pd.DataFrame(rows)
df_transactions['date'] = pd.to_datetime(df_transactions['date'])

periods = {
    'prepilot': {'begin': '2024-01-01', 'end': '2024-01-14'},
    'pilot': {'begin': '2024-01-15', 'end': '2024-01-28'}
}

result = calculate_metric_cuped(
    df_transactions, 'revenue', 'user_id', user_ids, 'date', periods, 'revenue'
)

print(f"Result shape: {result.shape}")
print(f"Columns: {list(result.columns)}")
print(f"\nFirst 5 rows:")
print(result.head())

# Verify structure
assert len(result) == n_users, f"Expected {n_users} rows, got {len(result)}"
assert 'metric' in result.columns, "Missing 'metric' column"
assert 'metric_prepilot' in result.columns, "Missing 'metric_prepilot' column"
assert 'metric_cuped' in result.columns, "Missing 'metric_cuped' column"

# The key test: CUPED should reduce variance
var_original = result['metric'].var()
var_cuped = result['metric_cuped'].var()
reduction = 1 - var_cuped / var_original

print(f"\nVariance of original metric: {var_original:.2f}")
print(f"Variance of CUPED metric:    {var_cuped:.2f}")
print(f"Variance reduction:           {reduction:.1%}")

assert var_cuped < var_original, "CUPED metric should have lower variance than original!"
print("\nAll Task 2 checks passed!")

---

## Task 3: Demonstrate Variance Reduction with a Simulated Experiment

### Context

Now prove to the QuickCart team that CUPED actually works in practice. Set up a simulated A/B test and show that:
1. The standard t-test on raw revenue has low power for a small effect
2. The same t-test on CUPED-adjusted revenue has much higher power

### Problem

Design a simulation that:
1. Generates synthetic transaction data for 2000 users (1000 control, 1000 treatment)
2. Adds a **true 2% lift** to the treatment group during the pilot period
3. Runs 500 simulated experiments, each time:
   - Computing raw pilot-period metrics for both groups
   - Computing CUPED-adjusted metrics for both groups
   - Running t-tests on both
4. Reports detection rates (power) for both approaches

**Target:** Show that CUPED increases power from ~20-30% to ~50-70% (or better) for this scenario.

<details>
<summary>Hint (click to expand)</summary>

Generate correlated data using a shared user-level random effect:
```python
# Each user has a stable spending level
user_baseline = np.random.lognormal(mean=3, sigma=1, size=n_users)

# Pre-period and pilot-period spending are both driven by this baseline
pre_revenue = user_baseline * np.random.exponential(1, n_users)
pilot_revenue = user_baseline * np.random.exponential(1, n_users)

# Treatment effect (2% lift) only in pilot period for treatment group
pilot_revenue[treatment_mask] *= 1.02
```

</details>

In [ ]:
# YOUR CODE HERE
# 
# Suggested structure:
# 1. Set parameters (n_users, n_simulations, true_effect)
# 2. Loop over simulations:
#    a. Generate pre-period and pilot-period data with correlation
#    b. Add treatment effect to pilot period for treatment group
#    c. Compute CUPED adjustment (theta, adjusted metric)
#    d. Run t-test on raw metric -> store p-value
#    e. Run t-test on CUPED metric -> store p-value
# 3. Calculate power = fraction of p-values < 0.05
# 4. Print comparison

n_users_per_group = 1000
n_simulations = 500
true_effect = 0.02  # 2% lift
alpha = 0.05

# YOUR CODE HERE
pass

In [ ]:
# Verify your results
# After running the simulation, you should have variables like:
# power_raw: detection rate without CUPED
# power_cuped: detection rate with CUPED

# Uncomment and run after completing the simulation:
# assert power_cuped > power_raw, "CUPED should increase power"
# assert power_cuped > 0.4, "CUPED power should be substantially higher than raw"
# print(f"Power without CUPED: {power_raw:.1%}")
# print(f"Power with CUPED:    {power_cuped:.1%}")
# print(f"Power improvement:   {power_cuped - power_raw:.1%}")
# print("\nTask 3 checks passed!")

---

## Task 4: Variance Reduction vs. Correlation

### Context

The theoretical variance reduction from CUPED is:

$$\text{Var}(Y_{\text{cuped}}) = \text{Var}(Y) \cdot (1 - \rho^2)$$

This means:
- $\rho = 0$: no reduction (CUPED is useless)
- $\rho = 0.5$: 25% reduction
- $\rho = 0.7$: 51% reduction
- $\rho = 0.9$: 81% reduction

Verify this empirically and help the QuickCart team understand when CUPED helps most.

### Problem

1. Generate data with varying correlation between pre-period and pilot-period metrics (e.g., $\rho \in \{0.1, 0.2, ..., 0.9\}$)
2. For each correlation level, compute the empirical variance reduction from CUPED
3. Plot: empirical variance reduction vs. theoretical $(1 - \rho^2)$ curve
4. Answer: What correlation did your Task 3 data have? What does this imply about how much pre-experiment data to collect?

<details>
<summary>Hint: Generating data with a specific correlation</summary>

Use the Cholesky decomposition or simply:
```python
from scipy.stats import multivariate_normal

# Generate bivariate normal with desired correlation
cov_matrix = [[1, rho], [rho, 1]]
data = multivariate_normal.rvs(mean=[0, 0], cov=cov_matrix, size=n)
pre_period = data[:, 0]
pilot_period = data[:, 1]
```

</details>

In [ ]:
# YOUR CODE HERE
#
# Suggested structure:
# 1. Define a range of target correlations: rhos = np.arange(0.1, 1.0, 0.1)
# 2. For each rho:
#    a. Generate correlated (pre_period, pilot_period) data
#    b. Compute theta and CUPED-adjusted values
#    c. Compute empirical variance reduction: 1 - Var(cuped) / Var(pilot)
# 3. Plot empirical vs theoretical (1 - rho^2) variance reduction

pass

In [ ]:
# Verification: your plot should show empirical points closely tracking the theoretical curve
# The theoretical curve is: variance_reduction = rho^2
#
# Uncomment after completing:
# assert len(empirical_reductions) >= 5, "Test at least 5 correlation levels"
# # Check that empirical roughly matches theoretical
# for rho, emp_reduction in zip(rhos, empirical_reductions):
#     theoretical = rho**2
#     assert abs(emp_reduction - theoretical) < 0.1, \
#         f"At rho={rho:.1f}, empirical ({emp_reduction:.3f}) too far from theoretical ({theoretical:.3f})"
# print("Task 4 checks passed! Empirical matches theory.")

**Your analysis:**

*Answer the following:*
1. *What correlation level existed in your Task 3 simulation?*
2. *At what correlation does CUPED reduce the required sample size by 50%?*
3. *What practical steps could QuickCart take to maximize correlation (and thus CUPED benefit)?*

---

## Summary

In this assignment you:
- Implemented the core metric aggregation used in experiment analysis
- Built a CUPED variance reduction pipeline
- Demonstrated empirically that CUPED increases statistical power
- Explored the relationship between pre/post correlation and variance reduction

**Key takeaway:** CUPED is essentially "free power" — it uses data you already have (pre-experiment behavior) to make your experiments more sensitive. The only requirement is that pre-period behavior is correlated with in-experiment behavior, which is almost always true for user-level metrics.